In [ ]:
# Fabric ML Experiment — LightGBM Demand Forecasting
# Input:  silver_features
# Output: MLflow registered model + silver_forecast_results# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
from lightgbm import (LGBMRegressor, early_stopping, log_evaluation)
from sklearn.metrics import (mean_absolute_error, mean_squared_error,mean_absolute_percentage_error)
from sklearn.model_selection import KFold

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 3, Finished, Available, Finished, False)

## Configure MLflow Experiment

In [2]:
# MLflow is built into Fabric — all runs are tracked
# automatically in the workspace experiment UI.
EXPERIMENT_NAME = "RetailDemandForecast_v1"
mlflow.set_experiment(EXPERIMENT_NAME)
 
# Product families to model (train one model per family)
# For full production: loop over all 33 families
FAMILIES_TO_TRAIN = ["PRODUCE", "GROCERY I", "BEVERAGES", "CLEANING", "DAIRY"]

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 4, Finished, Available, Finished, False)

2026/04/25 08:40:56 INFO mlflow.tracking.fluent: Experiment with name 'RetailDemandForecast_v1' does not exist. Creating a new experiment.


## Feature Definition

In [3]:
FEATURE_COLS = [
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_364",
    "roll_avg_7", "roll_avg_14", "roll_avg_28",
    "roll_std_7", "roll_std_14",
    "roll_max_7", "roll_min_7", "roll_median_7", "trend_7v14",
    "onpromotion", "promo_lag_1", "promo_lag_7",
    "oil_price", "is_holiday",
    "store_day_total", "store_day_txn_total",
    "family_share_pct", "cluster_avg_sales",
    "year", "month", "day", "weekday", "week_of_year",
    "quarter", "is_month_end", "is_month_start",
    "family_idx", "store_type_idx", "city_idx", "cluster",
]
TARGET = "sales"

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 5, Finished, Available, Finished, False)

## Model Hyperparameters

In [4]:
LGBM_PARAMS = {
    "n_estimators":       1000,
    "learning_rate":      0.03,
    "max_depth":          8,
    "num_leaves":         63,
    "min_child_samples":  20,
    "subsample":          0.8,
    "colsample_bytree":   0.8,
    "reg_alpha":          0.1,
    "reg_lambda":         0.1,
    "random_state":       42,
    "n_jobs":             -1,
    "verbose":            -1,
}

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 6, Finished, Available, Finished, False)

## Training Loop (one model per family)

In [5]:
all_predictions = []
results_summary = []
 
df_features = spark.read.format("delta").table("silver_features")
 
for FAMILY in FAMILIES_TO_TRAIN:
    print(f"\n{'='*40}")
    print(f"Training: {FAMILY}")
    print(f"{'='*40}")
 
    # --- Load family subset ---
    df_pd = (
        df_features
        .filter(F.col("family") == FAMILY)
        .select(["date", "store_nbr", TARGET] + FEATURE_COLS)
        .toPandas()
        .sort_values("date")
        .reset_index(drop=True)
    )
 
    # --- Temporal train/test split (last 28 days = test) ---
    cutoff   = df_pd["date"].max() - pd.Timedelta(days=28)
    df_train = df_pd[df_pd["date"] <= cutoff].copy()
    df_test  = df_pd[df_pd["date"] >  cutoff].copy()
 
    X_train = df_train[FEATURE_COLS].fillna(0)
    y_train = df_train[TARGET]
    X_test  = df_test[FEATURE_COLS].fillna(0)
    y_test  = df_test[TARGET]
 
    print(f"  Train: {len(df_train):,} rows | Test: {len(df_test):,} rows")
 
    # --- Train with MLflow tracking ---
    with mlflow.start_run(run_name=f"LightGBM_{FAMILY.replace(' ','_')}_v1"):
 
        model = LGBMRegressor(**LGBM_PARAMS)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            callbacks=[early_stopping(80), log_evaluation(200)]
        )
 
        # Predictions (clip negatives — no negative sales)
        preds = np.clip(model.predict(X_test), 0, None)
 
        # Metrics
        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mape = mean_absolute_percentage_error(y_test + 1e-6, preds + 1e-6) * 100
 
        # Log everything to MLflow
        mlflow.log_param("family",        FAMILY)
        mlflow.log_param("train_rows",    len(df_train))
        mlflow.log_param("test_rows",     len(df_test))
        mlflow.log_param("n_features",    len(FEATURE_COLS))
        mlflow.log_params({f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()})
        mlflow.log_metric("MAE",          mae)
        mlflow.log_metric("RMSE",         rmse)
        mlflow.log_metric("MAPE_pct",     mape)
        mlflow.log_metric("best_iteration", model.best_iteration_)
 
        # Log feature importance
        fi = pd.DataFrame({
            "feature":   FEATURE_COLS,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        fi_path = f"/tmp/feature_importance_{FAMILY.replace(' ','_')}.csv"
        fi.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path, artifact_path="feature_importance")
 
        # Register the model
        mlflow.sklearn.log_model(
            model,
            artifact_path=f"lgbm_{FAMILY.lower().replace(' ','_')}",
            registered_model_name=f"RetailForecast_{FAMILY.replace(' ','_')}",
        )
 
        print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.1f}%")
        print(f"  Best iteration: {model.best_iteration_}")
        print(f"  Top 5 features: {fi['feature'].head(5).tolist()}")
 
    # --- Collect predictions for saving ---
    df_test = df_test.copy()
    df_test["predicted_sales"] = preds
    df_test["family"] = FAMILY
    all_predictions.append(
        df_test[["date","store_nbr","family",
                  "sales","predicted_sales"]]
    )
    results_summary.append({
        "family": FAMILY, "MAE": mae, "RMSE": rmse, "MAPE_pct": mape
    })

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 7, Finished, Available, Finished, False)


Training: PRODUCE
  Train: 87,912 rows | Test: 1,512 rows
Training until validation scores don't improve for 80 rounds
[200]	valid_0's l2: 12463.4
[400]	valid_0's l2: 7745.87
[600]	valid_0's l2: 6129.51
[800]	valid_0's l2: 5433.23
[1000]	valid_0's l2: 5055.02
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 5055.02
  MAE=38.06  RMSE=71.10  MAPE=2.2%
  Best iteration: 1000
  Top 5 features: ['store_day_total', 'family_share_pct', 'cluster_avg_sales', 'lag_7', 'lag_14']


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'RetailForecast_PRODUCE'.



Training: GROCERY I
  Train: 87,912 rows | Test: 1,512 rows
Training until validation scores don't improve for 80 rounds
[200]	valid_0's l2: 17377.3
[400]	valid_0's l2: 11849.3
[600]	valid_0's l2: 9857.71
[800]	valid_0's l2: 8848.36
[1000]	valid_0's l2: 8232.77
Did not meet early stopping. Best iteration is:
[998]	valid_0's l2: 8228.65
  MAE=50.82  RMSE=90.71  MAPE=1.1%
  Best iteration: 998
  Top 5 features: ['store_day_total', 'family_share_pct', 'cluster_avg_sales', 'store_day_txn_total', 'lag_28']



Training: BEVERAGES
  Train: 87,912 rows | Test: 1,512 rows
Training until validation scores don't improve for 80 rounds
[200]	valid_0's l2: 16872.8
[400]	valid_0's l2: 14779
[600]	valid_0's l2: 14157.1
Early stopping, best iteration is:
[649]	valid_0's l2: 14095.7
  MAE=51.10  RMSE=118.73  MAPE=1.3%
  Best iteration: 649
  Top 5 features: ['store_day_total', 'family_share_pct', 'cluster_avg_sales', 'lag_7', 'store_day_txn_total']



Training: CLEANING
  Train: 87,912 rows | Test: 1,512 rows
Training until validation scores don't improve for 80 rounds
[200]	valid_0's l2: 37199.9
Early stopping, best iteration is:
[124]	valid_0's l2: 36269.4
  MAE=76.82  RMSE=190.45  MAPE=5.6%
  Best iteration: 124
  Top 5 features: ['family_share_pct', 'store_day_total', 'cluster_avg_sales', 'lag_28', 'store_day_txn_total']



Training: DAIRY
  Train: 87,912 rows | Test: 1,512 rows
Training until validation scores don't improve for 80 rounds
[200]	valid_0's l2: 1037.66
[400]	valid_0's l2: 573.819
[600]	valid_0's l2: 440.288
[800]	valid_0's l2: 392.074
[1000]	valid_0's l2: 369.49
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 369.49
  MAE=10.82  RMSE=19.22  MAPE=1.3%
  Best iteration: 1000
  Top 5 features: ['store_day_total', 'family_share_pct', 'lag_7', 'lag_14', 'cluster_avg_sales']


## Save All Predictions to Delta Table

In [6]:
df_all_preds = pd.concat(all_predictions, ignore_index=True)
df_spark_preds = spark.createDataFrame(df_all_preds)
 
df_spark_preds.write.format("delta").mode("overwrite") \
              .option("overwriteSchema","true") \
              .saveAsTable("silver_forecast_results")
 
print(f"\n silver_forecast_results: {df_spark_preds.count():,} rows")

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 8, Finished, Available, Finished, False)


 silver_forecast_results: 7,560 rows


## Results Summary

In [7]:
print("\n" + "="*55)
print("MODEL PERFORMANCE SUMMARY")
print("="*55)
print(f"{'Family':<20} {'MAE':>8} {'RMSE':>8} {'MAPE%':>8}")
print("-"*55)
for r in results_summary:
    print(f"{r['family']:<20} {r['MAE']:>8.2f} {r['RMSE']:>8.2f} {r['MAPE_pct']:>7.1f}%")
 
print("\nView full experiment in: Fabric Workspace → Experiments →", EXPERIMENT_NAME)
print("Proceed to 05_gold_layer.sql in Fabric Warehouse")

StatementMeta(, 3d166834-abb1-4e47-8248-26040940f69c, 9, Finished, Available, Finished, False)


MODEL PERFORMANCE SUMMARY
Family                    MAE     RMSE    MAPE%
-------------------------------------------------------
PRODUCE                 38.06    71.10     2.2%
GROCERY I               50.82    90.71     1.1%
BEVERAGES               51.10   118.73     1.3%
CLEANING                76.82   190.45     5.6%
DAIRY                   10.82    19.22     1.3%

View full experiment in: Fabric Workspace → Experiments → RetailDemandForecast_v1
Proceed to 05_gold_layer.sql in Fabric Warehouse
